In [1]:
# currently used on blpc2 despite the name

import numpy as np 
import matplotlib.pyplot as plt 
import glob
import os
import pandas as pd
import multiprocessing
import logging
import blimpy as bl
%matplotlib inline

In [ ]:
df = pd.read_csv('/datax/scratch/benjb/bl_nearby_stars/BL_cadences_unique_nearby_star_sample_only.csv')
df.insert(0, column='Index', value=np.arange(len(df)))

unspliced_nums = []
for i, h5 in enumerate(df['.h5 path 1']):
    if not 'spliced' in h5:
        unspliced_nums.append(i)

pool_dats = np.sort(glob.glob('/datax/scratch/benjb/bl_nearby_stars/bliss_dats_pooled_090825/*.dat'))
skip_nums = []
for dat in pool_dats:
    n = int(os.path.basename(dat).split('_')[0])
    skip_nums.append(n)
skip_nums = np.unique(skip_nums)
skip_nums = np.concatenate([unspliced_nums, skip_nums])
skip_nums = unspliced_nums
skip_nums = np.sort(skip_nums)

check_idx = np.array([0, 1, 2, 63, 64, 65, 95, 96, 97, 121, 122, 123, 147, 148, 149,
             156, 157, 171, 172, 9178, 9179, 9180, 9197, 9198, 9199, 9200, 9201, 9202, 9203, 9204,
             9205, 9206, 9207, 9208, 9209, 9210, 9241, 9242, 19474, 19475, 19499, 19571, 19578, 19579, 
             19580, 19581, 19582, 19583, 19584, 19585, 28806, 28807, 28831, 28832, 28856, 28857, 28858, 
             28882, 28883, 28884, 28885, 28886, 28887, 28917, 28918, 28919, 28920, 28921, 28922, 28923])

check_idx = np.load('/datax/scratch/benjb/bl_nearby_stars/blpc2_spliced_files_idx_092925.npz')['arr_0']
check_idx = check_idx[check_idx > 26947]
print(check_idx)
print(len(check_idx))

# skip_nums = np.concatenate([unspliced_nums, skip_idx])
# skip_nums = np.sort(skip_nums)

print(len(df))

# NEED TO RUN THIS SEARCH ON ALL FILES IN SPLICED DIRECTORY -- FIGURE OUT WHICH IDX THESE ARE
df = df.iloc[check_idx]
#df.drop(index=skip_nums, inplace=True)

print(len(df))

[26948 26949 26950 ... 39081 39082 39113]
1397
39177
1397


In [ ]:
### FOR SETICORE

'''
logger = logging.getLogger(__name__)

def process_files(xxx):
    n = xxx[0]
    list_of_h5_files = xxx[1]
    # Remove all handlers associated with the root logger object.
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)
    logging.basicConfig(filename=f'{outdir_spare}000_{n}_051925.log', filemode="a", level=logging.DEBUG)
    for i, file in enumerate(list_of_h5_files):
        # Execute seticore in the terminal
        logger.info(f'Searching #{i+1} of {len(list_of_h5_files)} files ...')
        console = 'CUDA_DEVICE_ORDER=PCI_BUS_ID CUDA_VISIBLE_DEVICES=3 seticore ' + file + ' -M 4 -s 10 --output ' + outdir_spare + os.path.basename(file)[:-2] + 'dat'
        os.system(console)

if __name__ == "__main__":
    # Define n sets of files
    batch_size = len(unsearched_h5_list)//20
    file_sets = [[str(i), unsearched_h5_list[i*batch_size:(i+1)*batch_size]] for i in range(20)]
    file_sets.append([str(20), unsearched_h5_list[20*batch_size:]])

    # Create a pool of n processes
    with multiprocessing.Pool(processes=21) as pool:
        # Map the process_files function to each file set
        pool.map(process_files, file_sets)

    print("All files processed.")
'''

In [ ]:
### FOR BLISS:

outdir = '/datax/scratch/benjb/bl_nearby_stars/bliss_dats_spliced_092925/'
logdir = '/datax/scratch/benjb/bl_nearby_stars/bliss_logs/'

logger = logging.getLogger(__name__)

def process_files(xxx):
    n = xxx[0]
    dfbatch = xxx[1]
    snr = 20
    #pfb = '/datax/scratch/benjb/bl_nearby_stars/GBT_spliced_PFB_response.f32'
    # Remove all handlers associated with the root logger object.
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)
    logging.basicConfig(filename=f'{logdir}0_2_0_{n}_052726.log', filemode="a", level=logging.DEBUG)
    # old_log = f'{outdir}000_{n}_060525.log'
    # with open(old_log) as f:
    #     f = f.readlines()
    # important = []
    # for line in f:
    #     if 'cadences' in line:
    #         important.append(line)
    # last_line = important[-1]
    # skip_number = int((last_line.split('#')[1]).split(' of ')[0])
    for i in range(len(dfbatch)):
        index = dfbatch.iloc[i]['Index']
        if index < 26384:
            logger.info(f'Skipping index {index} prior to 26384 ...')
            continue
        logger.info(f'Searching #{i+1} of {len(dfbatch)} cadences (index {index} of 39177 overall) ...')
        # if i+1 < skip_number:
        #     logger.info('Already searched this cadence. Continuing ...')
        #     continue
        # if i+1 in skip_nums:
        #     logger.info('Already searched this cadence. Continuing ...')
        #     continue
        test_file = dfbatch[f'.h5 path 1'].values[i]
        fb = bl.Waterfall(test_file, load_data=False)
        nfc = fb.header['nchans']
        # check for configuration
        if nfc % (2**20) == 0:
            # configuration is normal
            ncc = nfc // 2**20
            nfpc = 2**20
            config = 'u' # usual
            pfb = '/datax/scratch/benjb/bl_nearby_stars/GBT_spliced_PFB_response.f32'
        elif nfc % (1033216) == 0:
            ncc = nfc // 1033216
            nfpc = 1033216
            config = 'o' # old
            pfb = '/datax/scratch/benjb/bl_nearby_stars/GBT_spliced_PFB_response_1033216.f32'
        else:
            print('Unusual configuration; skipping for now.')
            continue
        for j in range(6): # for each h5 file in a cadence
            h5idx = j+1
            file = dfbatch[f'.h5 path {h5idx}'].values[i]
            logger.info(f'  Searching #{h5idx} of 6 files in this cadence ({file}) ...')
            if not 'spliced' in file:
                logger.info('  Unspliced file! Continuing ...')
                continue
            # run BLISS
            # try:
            #     console = f'bliss_find_hits {file} -e {pfb} -d cuda:{n} -md -4 -MD 4 -s {snr} --number-coarse 64 --distance 30 --output ' + outdir + f'{index}_{h5idx}_' + os.path.basename(file)[:-3] + f'_nosig_nosk_SNR_{snr}_L1_30.dat'
            #     os.system(console)
            # run BLISS for each coarse channel
            for k in range(ncc):
                # check whether dat already exists:
                datcheck = outdir + f'{index}_{h5idx}_{k}_' + os.path.basename(file)[:-3] + f'_nosig_nosk_SNR_{snr}_L1_30.dat'
                if os.path.exists(datcheck):
                    with open(datcheck) as f:
                        lines = f.readlines()
                    if len(lines) > 0:
                        if k % 200 == 0:
                            logger.info(f'  Channel {k} of {ncc}: Already searched!')
                    else:
                        logger.info(f'  Channel {k} of {ncc}: Empty .dat. Re-searching ...')
                        try: # for spliced files, do one cc at a time
                            if k%200 == 0:
                                logger.info(f'  Channel {k} of {ncc}: Searching ...')
                            console = f'bliss_find_hits {file} -e {pfb} -d cuda:{n} -md -4 -MD 4 -s {snr} --number-coarse 1 -c {k} --nchan-per-coarse {nfpc} --distance 30 --output ' + outdir + f'{index}_{h5idx}_{k}_' + os.path.basename(file)[:-3] + f'_nosig_nosk_SNR_{snr}_L1_30.dat'
                            os.system(console)
                        except:
                            print('Search failed; check later.')
                if not os.path.exists(datcheck):
                    try: # for spliced files, do one cc at a time
                        if k%200 == 0:
                            logger.info(f'  Channel {k} of {ncc}: Searching ...')
                        console = f'bliss_find_hits {file} -e {pfb} -d cuda:{n} -md -4 -MD 4 -s {snr} --number-coarse 1 -c {k} --nchan-per-coarse {nfpc} --distance 30 --output ' + outdir + f'{index}_{h5idx}_{k}_' + os.path.basename(file)[:-3] + f'_nosig_nosk_SNR_{snr}_L1_30.dat'
                        os.system(console)
                    except:
                        print('Search failed; check later.')
                else:
                    if k % 200 == 0:
                        logger.info(f'  Channel {k} of {ncc}: Already searched!')

if __name__ == "__main__":
    # Define n sets of files
    nnodes = 4
    # batch_size = len(df) // 3 // nnodes
    # file_sets = [[str(i), df.iloc[i*batch_size+len(df)*2//3:(i+1)*batch_size+len(df)*2//3]] for i in range(nnodes)] # run on all 4 GPUs

    batch_size = len(df) // nnodes 
    file_sets = [[str(i), df.iloc[i*batch_size:(i+1)*batch_size]] for i in range(nnodes)]

    #file_sets = [[str(i), df.iloc[i*batch_size+len(df)*2//3:(i+1)*batch_size+len(df)*2//3]] for i in range(nnodes-1)] # leave a GPU open
    #file_sets = [[str(i), df.iloc[i*batch_size:(i+1)*batch_size]] for i in range(nnodes)]
    #file_sets.append([str(nnodes), df.iloc[nnodes*batch_size+len(df)*3//4:len(df)]])

    # Create a pool of n processes
    with multiprocessing.Pool(processes=nnodes) as pool:
        # Map the process_files function to each file set
        pool.map(process_files, file_sets)

    print("All files processed.")

In [ ]:
# console = 'export CUDA_DEVICE_ORDER=PCI_BUS_ID'
# os.system(console)
# console = 'export CUDA_VISIBLE_DEVICES=3'
# os.system(console)

# for i, file in enumerate(h5list):
#     # Execute seticore in the terminal
#     print(f'Searching #{i+1} of {len(h5list)} files ...')
#     check_dat = outdir + os.path.basename(file)[:-2] + 'dat'
#     if os.path.exists(check_dat):
#         print('Already searched!')
#         continue
#     console = 'CUDA_DEVICE_ORDER=PCI_BUS_ID CUDA_VISIBLE_DEVICES=3 seticore ' + file + ' -M 4 -s 10 --output ' + outdir + os.path.basename(file)[:-2] + 'dat'
#     os.system(console)

Searching #1 of 168 files ...
Searching #2 of 168 files ...
Searching #3 of 168 files ...
Searching #4 of 168 files ...
Searching #5 of 168 files ...
Searching #6 of 168 files ...
Searching #7 of 168 files ...
Searching #8 of 168 files ...
Searching #9 of 168 files ...
Searching #10 of 168 files ...
Searching #11 of 168 files ...
Searching #12 of 168 files ...
Searching #13 of 168 files ...
Searching #14 of 168 files ...
Searching #15 of 168 files ...
Searching #16 of 168 files ...
Searching #17 of 168 files ...
Searching #18 of 168 files ...
Searching #19 of 168 files ...
Searching #20 of 168 files ...
Searching #21 of 168 files ...
Searching #22 of 168 files ...
Searching #23 of 168 files ...
Searching #24 of 168 files ...
Searching #25 of 168 files ...
Searching #26 of 168 files ...
Searching #27 of 168 files ...
Searching #28 of 168 files ...
Searching #29 of 168 files ...
Searching #30 of 168 files ...
Searching #31 of 168 files ...
Searching #32 of 168 files ...
Searching #33 of 

In [ ]:
'/datax/scratch/benjb/bl_nearby_stars/seticore_output/beyond_5.1_pc/spliced_blc0001020304050607_guppi_57692_59443_HIP58576_0019.gpuspec.0000.dat'

In [ ]:
#from turbo_seti.find_doppler.find_doppler import FindDoppler
#file = '/datag/pipeline/AGBT23B_999_31/blc06_blp06/blc06_guppi_60331_81094_HIP649_0130.rawspec.0000.h5'

#doppler = FindDoppler(file,
#                      max_drift = 4,
#                      snr = 10,       
#                      out_dir = outdir+'turboSETI/',
#                      n_coarse_chan = 2000,
#                      gpu_backend = True,
#                      blank_dc = True
#                     )
#doppler.search()


turbo_seti version 2.3.2
blimpy version 2.1.4
h5py version 2.10.0
hdf5plugin version 2.1.2
HDF5 library version 1.10.5




OSError: File /datag/pipeline/AGBT23B_999_31/blc06_blp06/blc06_guppi_60331_81094_HIP649_0130.rawspec.0000.h5 doesn't exist, please check!